In [4]:
# phase3_content_generation_robust.py

import os
import gspread
from openai import OpenAI
from dotenv import load_dotenv
import re
import time

load_dotenv()

# ---------------------
# Setup
# ---------------------
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("⚠️ Please set your OPENAI_API_KEY in environment variables")

client = OpenAI(api_key=OPENAI_API_KEY)

# Connect to Google Sheets
gc = gspread.service_account(filename="credentials.json")
sheet = gc.open("Job_Trends_Agent").sheet1

# ---------------------
# GPT-2 Prompt Template
# ---------------------
def generate_content(trend, category):
    prompt = f"""
You are a content creator. Based on the trend: "{trend}" 
(category: {category}), generate the following clearly labeled sections:

Instagram Post:
- Short caption (1–2 lines)
- 3–5 relevant hashtags

Blog Draft:
- 2–3 paragraph blog-style draft
- Include a placeholder link for this category:
  - Admit Card → [Admit Card Link]
  - Job Notification → [Job Notification Link]
  - Result → [Result Link]

YouTube Reel:
- One-liner catchy caption
- 3–5 hashtags

Thumbnail Idea:
- Short catchy text for thumbnail
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ GPT generation failed for '{trend}': {e}")
        return None

# ---------------------
# Parse GPT content reliably
# ---------------------
def parse_content(text):
    sections = {"Instagram": "", "Blog": "", "Reel": "", "Thumbnail": ""}
    patterns = {
        "Instagram": r"Instagram Post:([\s\S]*?)\n(?:Blog Draft:|$)",
        "Blog": r"Blog Draft:([\s\S]*?)\n(?:YouTube Reel:|$)",
        "Reel": r"YouTube Reel:([\s\S]*?)\n(?:Thumbnail Idea:|$)",
        "Thumbnail": r"Thumbnail Idea:([\s\S]*)"
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            sections[key] = match.group(1).strip()
    
    return sections

# ---------------------
# Main Logic
# ---------------------
def update_content():
    rows = sheet.get_all_values()
    header, data = rows[0], rows[1:]

    for i, row in enumerate(data, start=2):  # start=2 because row 1 = header
        trend, category = row[0], row[1]

        # Skip if no category or already filled
        if not category or any(row[2:6]):  
            continue  

        print(f"🔄 Generating content for: {trend} ({category})")

        content = generate_content(trend, category)
        if not content:
            continue

        parsed = parse_content(content)

        # Update sheet (cols C–F = Instagram, Blog, Reel, Thumbnail)
        try:
            sheet.update(f"C{i}:F{i}", [[
                parsed["Instagram"],
                parsed["Blog"],
                parsed["Reel"],
                parsed["Thumbnail"]
            ]])
            print(f"✅ Updated row {i} in Google Sheet")
        except Exception as e:
            print(f"⚠️ Failed to update Google Sheet for row {i}: {e}")
        
        # Optional: small delay to avoid rate limits
        time.sleep(1)

if __name__ == "__main__":
    update_content()
    print("🎉 Phase 3 complete! Content generated and added to Google Sheet.")


🎉 Phase 3 complete! Content generated and added to Google Sheet.
